In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# 读取数据
print("正在读取数据...")
df = pd.read_csv(r"C:\Users\lenovo\Desktop\期中数据\ruc_Class25Q2_train_price.csv")
print(f"数据形状: {df.shape}")

def plate_based_geoclustering(df, n_clusters_range=(3, 10), min_samples_per_plate=5):
    """
    基于板块价格特征的地理聚类
    返回聚类结果和详细的板块-聚类映射关系
    """
    print("="*60)
    print("开始基于板块价格的地理聚类分析")
    print("="*60)
    
    # 1. 计算每个板块的价格统计特征
    print("\n1. 计算板块价格统计特征...")
    plate_stats = df.groupby('板块').agg({
        'Price': ['mean', 'median', 'std', 'count', 'min', 'max']
    }).round(2)
    
    # 扁平化列名
    plate_stats.columns = ['价格均值', '价格中位数', '价格标准差', '样本数量', '最低价格', '最高价格']
    plate_stats = plate_stats.reset_index()
    
    # 2. 过滤样本量太少的板块
    print(f"\n2. 过滤样本量少于{min_samples_per_plate}的板块...")
    valid_plates = plate_stats[plate_stats['样本数量'] >= min_samples_per_plate].copy()
    invalid_plates = plate_stats[plate_stats['样本数量'] < min_samples_per_plate]['板块'].tolist()
    
    print(f"   - 总板块数: {len(plate_stats)}")
    print(f"   - 有效板块数: {len(valid_plates)}")
    print(f"   - 无效板块数(样本不足): {len(invalid_plates)}")
    
    if len(valid_plates) < 3:
        print("错误: 有效板块数量不足，无法进行聚类")
        return df, {}, {}
    
    # 3. 选择聚类特征并标准化
    print("\n3. 准备聚类特征...")
    clustering_features = ['价格均值', '价格标准差']  # 使用均值和标准差
    
    scaler = StandardScaler()
    X = scaler.fit_transform(valid_plates[clustering_features])
    
    # 4. 使用肘部法则确定最佳聚类数量
    print("\n4. 确定最佳聚类数量...")
    wcss = []
    k_range = range(n_clusters_range[0], min(n_clusters_range[1] + 1, len(valid_plates)))
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        wcss.append(kmeans.inertia_)
    
    # 绘制肘部图
    plt.figure(figsize=(10, 6))
    plt.plot(k_range, wcss, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('聚类数量 (k)', fontsize=12)
    plt.ylabel('WCSS (Within-Cluster Sum of Squares)', fontsize=12)
    plt.title('肘部法则 - 选择最佳聚类数量', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.savefig('板块聚类_肘部法则.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 自动选择拐点
    optimal_k = find_elbow_point(wcss, k_range)
    print(f"   - 自动选择聚类数量: {optimal_k}")
    
    # 5. 执行K-means聚类
    print(f"\n5. 执行K-means聚类 (k={optimal_k})...")
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    
    # 6. 创建板块到聚类的映射
    plate_to_cluster = dict(zip(valid_plates['板块'], cluster_labels))
    
    # 7. 应用到原始数据
    print("\n6. 将聚类结果应用到原始数据...")
    df_result = df.copy()
    df_result['板块价格聚类'] = df_result['板块'].map(plate_to_cluster)
    df_result['板块价格聚类'] = df_result['板块价格聚类'].fillna(-1)  # 无效板块标记为-1
    
    # 8. 创建详细的聚类映射记录
    print("\n7. 创建详细的聚类映射记录...")
    cluster_mapping_details = {}
    cluster_plate_lists = {}
    
    # 为每个聚类创建板块列表
    for cluster_id in range(optimal_k):
        plates_in_cluster = [plate for plate, cluster in plate_to_cluster.items() if cluster == cluster_id]
        cluster_plate_lists[f'聚类_{cluster_id}'] = plates_in_cluster
        cluster_mapping_details[f'聚类_{cluster_id}'] = {
            '板块数量': len(plates_in_cluster),
            '板块列表': plates_in_cluster,
            '平均价格': valid_plates[valid_plates['板块'].isin(plates_in_cluster)]['价格均值'].mean()
        }
    
    # 添加无效板块记录
    cluster_plate_lists['无效板块(样本不足)'] = invalid_plates
    cluster_mapping_details['无效板块'] = {
        '板块数量': len(invalid_plates),
        '板块列表': invalid_plates,
        '平均价格': 'N/A'
    }
    
    return df_result, cluster_mapping_details, cluster_plate_lists

def find_elbow_point(wcss, k_range):
    """自动找到肘部拐点"""
    # 计算二阶差分找到最大曲率点
    differences = []
    for i in range(1, len(wcss)-1):
        diff = (wcss[i-1] - wcss[i]) - (wcss[i] - wcss[i+1])
        differences.append(diff)
    
    if differences:
        elbow_index = differences.index(max(differences)) + 1
        return k_range[elbow_index]
    else:
        # 如果没有找到明显拐点，选择中间值
        return k_range[len(k_range)//2]

def analyze_clustering_results(df_with_clusters, cluster_mapping_details):
    """分析聚类结果"""
    print("\n" + "="*60)
    print("聚类结果分析")
    print("="*60)
    
    # 1. 基本统计
    cluster_counts = df_with_clusters['板块价格聚类'].value_counts().sort_index()
    print("\n各聚类样本数量分布:")
    for cluster_id, count in cluster_counts.items():
        if cluster_id == -1:
            print(f"  无效板块: {count:>6} 个样本")
        else:
            print(f"  聚类 {cluster_id}: {count:>6} 个样本")
    
    # 2. 价格统计
    print("\n各聚类价格统计:")
    cluster_stats = df_with_clusters.groupby('板块价格聚类')['Price'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).round(2)
    
    cluster_stats['占比(%)'] = (cluster_stats['count'] / len(df_with_clusters) * 100).round(2)
    print(cluster_stats)
    
    # 3. 聚类质量评估
    print("\n聚类质量评估:")
    total_variance = df_with_clusters['Price'].var()
    
    # 计算组内方差
    within_cluster_variance = 0
    for cluster_id in df_with_clusters['板块价格聚类'].unique():
        cluster_data = df_with_clusters[df_with_clusters['板块价格聚类'] == cluster_id]['Price']
        if len(cluster_data) > 1:
            within_cluster_variance += cluster_data.var() * len(cluster_data)
    
    within_cluster_variance = within_cluster_variance / len(df_with_clusters)
    between_cluster_r2 = 1 - (within_cluster_variance / total_variance)
    
    print(f"  总方差: {total_variance:.2f}")
    print(f"  组内方差: {within_cluster_variance:.2f}")
    print(f"  组间解释方差: {between_cluster_r2:.1%}")
    
    return cluster_stats, between_cluster_r2

def save_clustering_results(df_with_clusters, cluster_mapping_details, cluster_plate_lists):
    """保存聚类结果和映射关系"""
    print("\n" + "="*60)
    print("保存聚类结果")
    print("="*60)
    
    # 1. 保存带聚类标签的完整数据
    df_with_clusters.to_csv('带板块聚类标签的完整数据.csv', index=False, encoding='utf-8-sig')
    print("✓ 保存带聚类标签的完整数据: '带板块聚类标签的完整数据.csv'")
    
    # 2. 保存详细的聚类映射关系
    mapping_df = pd.DataFrame.from_dict(cluster_mapping_details, orient='index')
    mapping_df.to_csv('板块聚类映射详情.csv', encoding='utf-8-sig')
    print("✓ 保存聚类映射详情: '板块聚类映射详情.csv'")
    
    # 3. 保存每个聚类的板块列表（便于后续调用）
    plate_lists_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in cluster_plate_lists.items()]))
    plate_lists_df.to_csv('各聚类板块列表.csv', index=False, encoding='utf-8-sig')
    print("✓ 保存各聚类板块列表: '各聚类板块列表.csv'")
    
    # 4. 创建简化的映射表（便于预测时使用）
    simple_mapping = []
    for cluster_name, details in cluster_mapping_details.items():
        if cluster_name != '无效板块':
            for plate in details['板块列表']:
                simple_mapping.append({
                    '板块': plate,
                    '聚类标签': cluster_name,
                    '聚类ID': int(cluster_name.split('_')[1])
                })
    
    simple_mapping_df = pd.DataFrame(simple_mapping)
    simple_mapping_df.to_csv('板块聚类简化映射表.csv', index=False, encoding='utf-8-sig')
    print("✓ 保存简化映射表: '板块聚类简化映射表.csv'")
    
    return simple_mapping_df

# 执行地理聚类
print("开始执行地理聚类...")
df_with_clusters, cluster_mapping_details, cluster_plate_lists = plate_based_geoclustering(
    df, n_clusters_range=(4, 10), min_samples_per_plate=5
)

# 分析聚类结果
cluster_stats, cluster_r2 = analyze_clustering_results(df_with_clusters, cluster_mapping_details)

# 保存结果
simple_mapping_df = save_clustering_results(df_with_clusters, cluster_mapping_details, cluster_plate_lists)

# 输出关键信息供后续使用
print("\n" + "="*60)
print("关键信息汇总")
print("="*60)
print(f"聚类数量: {len([k for k in cluster_mapping_details.keys() if k != '无效板块'])}")
print(f"聚类解释方差: {cluster_r2:.1%}")
print(f"总板块数: {df['板块'].nunique()}")
print(f"有效聚类板块数: {len(simple_mapping_df)}")

# 显示每个聚类的简要信息
print("\n各聚类简要信息:")
for cluster_name, details in cluster_mapping_details.items():
    if cluster_name != '无效板块':
        avg_price = details['平均价格']
        plate_count = details['板块数量']
        print(f"  {cluster_name}: {plate_count:>2}个板块, 平均价格: {avg_price:.0f}元")

print("\n地理聚类完成! 结果已保存，可用于后续价格预测。")

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_predict
import warnings
import re
warnings.filterwarnings('ignore')

# 全局变量
train_feature_info = {}

def load_geocluster_mapping():
    """加载地理聚类映射表"""
    try:
        mapping_df = pd.read_csv('板块聚类简化映射表.csv')
        print(f"成功加载地理聚类映射表，包含 {len(mapping_df)} 个板块")
        return mapping_df
    except FileNotFoundError:
        print("警告: 未找到地理聚类映射表，将使用默认聚类")
        return None

def extract_numeric_from_string(s):
    """从字符串中提取数值"""
    if pd.isna(s):
        return 0
    # 匹配数字，包括小数
    numbers = re.findall(r'\d+\.?\d*', str(s))
    if numbers:
        return float(numbers[0])
    return 0

def process_building_area(area_str):
    """处理建筑面积，去掉单位并转换为数值"""
    if pd.isna(area_str):
        return 0
    # 去掉"㎡"单位并提取数值
    area_str = str(area_str).replace('㎡', '').strip()
    return extract_numeric_from_string(area_str)

def process_percentage(percent_str):
    """处理百分比数值"""
    if pd.isna(percent_str):
        return 0
    percent_str = str(percent_str).replace('%', '').strip()
    return extract_numeric_from_string(percent_str)

def process_floor(floor_str):
    """处理楼层信息，去掉括号说明并分类"""
    if pd.isna(floor_str):
        return '其他'
    
    # 去掉括号及后面的内容
    floor_str = str(floor_str).split('(')[0].strip().lower()
    
    if '地下室' in floor_str or '地下' in floor_str:
        return '地下室'
    elif '底层' in floor_str or '底楼' in floor_str:
        return '底层'
    elif '低楼层' in floor_str or '低层' in floor_str:
        return '低楼层'
    elif '中楼层' in floor_str or '中层' in floor_str:
        return '中楼层'
    elif '高楼层' in floor_str or '高层' in floor_str:
        return '高楼层'
    elif '顶层' in floor_str or '顶楼' in floor_str:
        return '顶层'
    else:
        return '其他'

def process_renovation(reno_str):
    """处理装修情况"""
    if pd.isna(reno_str):
        return '其他'
    
    reno_str = str(reno_str).lower()
    if '精装' in reno_str:
        return '精装'
    elif '简装' in reno_str:
        return '简装'
    elif '毛坯' in reno_str:
        return '毛坯'
    elif '豪华' in reno_str:
        return '豪华装修'
    elif '中等' in reno_str:
        return '中等装修'
    else:
        return '其他'

def create_features_from_raw_data(df, is_training=True):
    """从原始数据创建特征"""
    processed_df = df.copy()
    
    print("从原始数据创建特征...")
    
    # 1. 处理建筑面积（关键步骤！）
    if '建筑面积' in processed_df.columns:
        processed_df['建筑面积_数值'] = processed_df['建筑面积'].apply(process_building_area)
        print(f"  - 建筑面积处理完成，范围: {processed_df['建筑面积_数值'].min():.1f} - {processed_df['建筑面积_数值'].max():.1f}")
    else:
        processed_df['建筑面积_数值'] = 0
        print("  - 警告: 未找到建筑面积列")
    
    # 2. 处理套内面积
    if '套内面积' in processed_df.columns:
        processed_df['套内面积_数值'] = processed_df['套内面积'].apply(process_building_area)
        # 计算得房率
        mask = (processed_df['建筑面积_数值'] > 0) & (processed_df['套内面积_数值'] > 0)
        processed_df['得房率'] = 0.0
        processed_df.loc[mask, '得房率'] = processed_df.loc[mask, '套内面积_数值'] / processed_df.loc[mask, '建筑面积_数值']
        processed_df['有套内面积'] = mask.astype(int)
        print(f"  - 套内面积处理完成，{mask.sum()} 个样本有套内面积")
    else:
        processed_df['套内面积_数值'] = 0
        processed_df['得房率'] = 0.0
        processed_df['有套内面积'] = 0
        print("  - 警告: 未找到套内面积列")
    
    # 3. 处理绿化率
    if '绿化率' in processed_df.columns:
        processed_df['绿化率_数值'] = processed_df['绿化率'].apply(process_percentage) / 100
        print(f"  - 绿化率处理完成，范围: {processed_df['绿化率_数值'].min():.3f} - {processed_df['绿化率_数值'].max():.3f}")
    elif '绿 化 率' in processed_df.columns:
        processed_df['绿化率_数值'] = processed_df['绿 化 率'].apply(process_percentage) / 100
        print(f"  - 绿化率处理完成，范围: {processed_df['绿化率_数值'].min():.3f} - {processed_df['绿化率_数值'].max():.3f}")
    else:
        processed_df['绿化率_数值'] = 0
        print("  - 警告: 未找到绿化率列")
    
    # 4. 处理容积率
    if '容积率' in processed_df.columns:
        processed_df['容积率_数值'] = processed_df['容积率'].apply(extract_numeric_from_string)
        print(f"  - 容积率处理完成，范围: {processed_df['容积率_数值'].min():.3f} - {processed_df['容积率_数值'].max():.3f}")
    elif '容 积 率' in processed_df.columns:
        processed_df['容积率_数值'] = processed_df['容 积 率'].apply(extract_numeric_from_string)
        print(f"  - 容积率处理完成，范围: {processed_df['容积率_数值'].min():.3f} - {processed_df['容积率_数值'].max():.3f}")
    else:
        processed_df['容积率_数值'] = 0
        print("  - 警告: 未找到容积率列")
    
    # 5. 处理是否有电梯
    elevator_columns = ['配备电梯', '有电梯', '电梯']
    elevator_found = False
    for col in elevator_columns:
        if col in processed_df.columns:
            processed_df['有电梯'] = processed_df[col].astype(str).str.contains('有|是|True|true|1', na=False).astype(int)
            elevator_count = processed_df['有电梯'].sum()
            print(f"  - 电梯信息处理完成，{elevator_count} 个样本有电梯")
            elevator_found = True
            break
    
    if not elevator_found:
        processed_df['有电梯'] = 0
        print("  - 警告: 未找到电梯信息列")
    
    # 6. 处理是否别墅
    villa_columns = ['别墅', '是别墅', '别墅类型']
    villa_found = False
    for col in villa_columns:
        if col in processed_df.columns:
            processed_df['是别墅'] = (~processed_df[col].isna() & (processed_df[col] != '')).astype(int)
            villa_count = processed_df['是别墅'].sum()
            print(f"  - 别墅信息处理完成，{villa_count} 个样本是别墅")
            villa_found = True
            break
    
    if not villa_found:
        processed_df['是别墅'] = 0
        print("  - 警告: 未找到别墅信息列")
    
    # 7. 处理产权信息
    if '产权所属' in processed_df.columns:
        processed_df['产权共有'] = processed_df['产权所属'].astype(str).str.contains('共有', na=False).astype(int)
        share_count = processed_df['产权共有'].sum()
        print(f"  - 产权信息处理完成，{share_count} 个样本是共有产权")
    else:
        processed_df['产权共有'] = 0
        print("  - 警告: 未找到产权信息列")
    
    # 8. 处理楼层信息
    if '所在楼层' in processed_df.columns:
        processed_df['楼层类型'] = processed_df['所在楼层'].apply(process_floor)
        # 创建楼层哑变量 - 确保所有可能的楼层类型都有对应的哑变量
        all_floor_types = ['地下室', '底层', '低楼层', '中楼层', '高楼层', '顶层', '其他']
        
        # 使用get_dummies创建哑变量
        floor_dummies = pd.get_dummies(processed_df['楼层类型'], prefix='楼层')
        
        # 确保所有楼层类型都有对应的列
        for floor_type in all_floor_types:
            col_name = f'楼层_{floor_type}'
            if col_name in floor_dummies.columns:
                processed_df[col_name] = floor_dummies[col_name]
            else:
                processed_df[col_name] = 0
        
        print(f"  - 楼层信息处理完成，类型分布: {dict(processed_df['楼层类型'].value_counts())}")
    else:
        processed_df['楼层类型'] = '其他'
        all_floor_types = ['地下室', '底层', '低楼层', '中楼层', '高楼层', '顶层', '其他']
        for floor_type in all_floor_types:
            processed_df[f'楼层_{floor_type}'] = 0
        print("  - 警告: 未找到楼层信息列")
    
    # 9. 处理装修情况
    if '装修情况' in processed_df.columns:
        processed_df['装修类型'] = processed_df['装修情况'].apply(process_renovation)
        # 创建装修哑变量 - 确保所有可能的装修类型都有对应的哑变量
        all_reno_types = ['精装', '简装', '毛坯', '豪华装修', '中等装修', '其他']
        
        # 使用get_dummies创建哑变量
        reno_dummies = pd.get_dummies(processed_df['装修类型'], prefix='装修')
        
        # 确保所有装修类型都有对应的列
        for reno_type in all_reno_types:
            col_name = f'装修_{reno_type}'
            if col_name in reno_dummies.columns:
                processed_df[col_name] = reno_dummies[col_name]
            else:
                processed_df[col_name] = 0
        
        print(f"  - 装修信息处理完成，类型分布: {dict(processed_df['装修类型'].value_counts())}")
    else:
        processed_df['装修类型'] = '其他'
        all_reno_types = ['精装', '简装', '毛坯', '豪华装修', '中等装修', '其他']
        for reno_type in all_reno_types:
            processed_df[f'装修_{reno_type}'] = 0
        print("  - 警告: 未找到装修信息列")
    
    # 10. 应用地理聚类
    mapping_df = load_geocluster_mapping()
    if mapping_df is not None and '板块' in processed_df.columns:
        plate_to_cluster = dict(zip(mapping_df['板块'], mapping_df['聚类ID']))
        processed_df['地理聚类'] = processed_df['板块'].map(plate_to_cluster)
        processed_df['地理聚类'] = processed_df['地理聚类'].fillna(-1)
        
        # 创建5个聚类哑变量
        for cluster_id in range(5):
            processed_df[f'地理聚类_{cluster_id}'] = (processed_df['地理聚类'] == cluster_id).astype(int)
        
        cluster_counts = processed_df['地理聚类'].value_counts()
        print(f"  - 地理聚类应用完成，分布: {dict(cluster_counts)}")
    else:
        print("  - 警告: 无法应用地理聚类")
        for cluster_id in range(5):
            processed_df[f'地理聚类_{cluster_id}'] = 0
    
    return processed_df

def prepare_features(df, is_training=True):
    """准备特征矩阵"""
    print(f"{'训练' if is_training else '测试'}特征准备...")
    
    # 从原始数据创建特征
    df_processed = create_features_from_raw_data(df, is_training)
    
    # 定义特征列
    feature_columns = []
    
    # 1. 地理聚类特征（5个）
    for i in range(5):
        feature_columns.append(f'地理聚类_{i}')
    
    # 2. 面积特征
    area_features = ['建筑面积_数值', '套内面积_数值', '得房率', '有套内面积']
    feature_columns.extend([f for f in area_features if f in df_processed.columns])
    
    # 3. 数值特征
    numeric_features = ['绿化率_数值', '容积率_数值']
    feature_columns.extend([f for f in numeric_features if f in df_processed.columns])
    
    # 4. 布尔特征
    bool_features = ['是别墅', '产权共有', '有电梯']
    feature_columns.extend([f for f in bool_features if f in df_processed.columns])
    
    # 5. 楼层哑变量（从df_processed中获取）
    floor_features = [f for f in df_processed.columns if f.startswith('楼层_')]
    feature_columns.extend(floor_features)
    
    # 6. 装修哑变量（从df_processed中获取）
    reno_features = [f for f in df_processed.columns if f.startswith('装修_')]
    feature_columns.extend(reno_features)
    
    # 创建特征矩阵
    X = pd.DataFrame(index=df_processed.index)
    
    # 确保所有特征都存在
    for feature in feature_columns:
        if feature in df_processed.columns:
            X[feature] = df_processed[feature]
        else:
            print(f"警告: 特征 '{feature}' 不存在，将用0填充")
            X[feature] = 0
    
    # 确保所有特征都是数值类型
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        if X[col].isna().any():
            X[col].fillna(0, inplace=True)
    
    print(f"{'训练' if is_training else '测试'}特征矩阵形状: {X.shape}")
    print(f"使用的特征数量: {len(feature_columns)}")
    
    return X, feature_columns

def train_model(X, y):
    """训练模型"""
    print("\n训练模型...")
    
    # 1. 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=111
    )
    
    print(f"  训练集样本数: {len(X_train)}")
    print(f"  测试集样本数: {len(X_test)}")
    
    # 2. 处理异常值
    price_lower = y_train.quantile(0.01)
    price_upper = y_train.quantile(0.99)
    
    valid_indices = (y_train >= price_lower) & (y_train <= price_upper)
    X_train_clean = X_train[valid_indices].copy()
    y_train_clean = y_train[valid_indices].copy()
    
    print(f"  清理后训练样本数: {len(y_train_clean)}")
    print(f"  移除异常值: {len(y_train) - len(y_train_clean)} 个")
    
    # 3. 对目标变量进行对数变换
    y_train_log = np.log1p(y_train_clean)
    y_test_log = np.log1p(y_test)
    
    # 4. 标准化特征
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled = scaler.transform(X_test)
    
    # 保存标准化器
    train_feature_info['scaler'] = scaler
    
    # 5. 定义模型
    models = {
        'OLS': LinearRegression(),
        'Ridge_0.1': Ridge(alpha=0.1),
        'Ridge_1.0': Ridge(alpha=1.0),
        'Ridge_10.0': Ridge(alpha=10.0),
        'LASSO_0.1': Lasso(alpha=0.1, max_iter=10000),
        'LASSO_1.0': Lasso(alpha=1.0, max_iter=10000),
        'ElasticNet_0.1': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
        'ElasticNet_1.0': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
    }
    
    results = {}
    
    # 6. 训练和评估模型
    for name, model in models.items():
        print(f"\n--- 训练 {name} 模型 ---")
        
        try:
            # 训练模型
            model.fit(X_train_scaled, y_train_log)
            
            # 样本内预测
            y_train_pred_log = model.predict(X_train_scaled)
            y_train_pred = np.expm1(y_train_pred_log)
            
            # 样本外预测
            y_test_pred_log = model.predict(X_test_scaled)
            y_test_pred = np.expm1(y_test_pred_log)
            
            # 计算指标
            train_mae = mean_absolute_error(y_train_clean, y_train_pred)
            train_r2 = r2_score(y_train_clean, y_train_pred)
            
            test_mae = mean_absolute_error(y_test, y_test_pred)
            test_r2 = r2_score(y_test, y_test_pred)
            
            # 交叉验证
            y_cv_pred_log = cross_val_predict(model, X_train_scaled, y_train_log, cv=6)
            y_cv_pred = np.expm1(y_cv_pred_log)
            cv_mae = mean_absolute_error(y_train_clean, y_cv_pred)
            
            # 检查预测值
            min_pred = min(y_train_pred.min(), y_test_pred.min())
            max_pred = max(y_train_pred.max(), y_test_pred.max())
            negative_count = np.sum(y_test_pred < 0)
            
            results[name] = {
                'model': model,
                'train_mae': train_mae,
                'train_r2': train_r2,
                'test_mae': test_mae,
                'test_r2': test_r2,
                'cv_mae': cv_mae,
                'min_pred': min_pred,
                'max_pred': max_pred,
                'negative_count': negative_count
            }
            
            print(f"  样本内 MAE: {train_mae:,.2f}")
            print(f"  样本外 MAE: {test_mae:,.2f}")
            print(f"  交叉验证 MAE: {cv_mae:,.2f}")
            print(f"  预测范围: {min_pred:,.2f} - {max_pred:,.2f}")
            print(f"  负值数量: {negative_count}")
            
        except Exception as e:
            print(f"  {name} 模型训练失败: {e}")
            results[name] = None
    
    return results, X_train_clean.columns.tolist()

def print_performance_table(results):
    """打印性能表格"""
    print("\n" + "="*80)
    print("性能汇总表格 (MAE指标)")
    print("="*80)
    print("| Metrics           | In sample     | out of sample | Cross-validation | Kaggle Score |")
    print("|-------------------|---------------|---------------|------------------|--------------|")
    
    # 确定最佳模型
    valid_results = {k: v for k, v in results.items() if v is not None and v['negative_count'] == 0}
    best_model_name = min(valid_results.keys(), key=lambda x: valid_results[x]['test_mae']) if valid_results else None
    
    # 主要模型显示
    main_models = {
        'OLS': 'OLS',
        'LASSO': 'LASSO_0.1',
        'Best Linear Model': best_model_name
    }
    
    for display_name, model_key in main_models.items():
        if model_key and model_key in results and results[model_key] is not None:
            result = results[model_key]
            print(f"| {display_name:17} | {result['train_mae']:12,.2f} | {result['test_mae']:12,.2f} | {result['cv_mae']:15,.2f} | {'-':12} |")
        else:
            print(f"| {display_name:17} | {'N/A':12} | {'N/A':12} | {'N/A':15} | {'-':12} |")
    
    print("="*80)

def predict_test_data(test_path, template_path, output_path, final_model, feature_names):
    """预测测试集数据"""
    print("\n处理外部测试集...")
    
    # 1. 读取测试数据和提交模板
    test_df = pd.read_csv(test_path)
    submission_df = pd.read_csv(template_path)
    
    print(f"测试数据形状: {test_df.shape}")
    print(f"提交模板形状: {submission_df.shape}")
    
    # 2. 根据提交模板中的ID筛选需要预测的测试数据
    template_ids = set(submission_df['ID'])
    test_df_filtered = test_df[test_df['ID'].isin(template_ids)].copy()
    
    print(f"筛选后需要预测的测试数据形状: {test_df_filtered.shape}")
    
    # 3. 准备测试特征
    X_test, _ = prepare_features(test_df_filtered, is_training=False)
    
    # 确保特征一致
    for feature in feature_names:
        if feature not in X_test.columns:
            print(f"警告: 特征 '{feature}' 在测试集中不存在，将用0填充")
            X_test[feature] = 0
    
    X_test = X_test[feature_names]
    
    # 4. 使用训练好的模型进行预测
    print("使用训练好的模型进行预测...")
    
    # 确保测试数据没有NaN值
    if X_test.isna().any().any():
        print("警告: 测试数据中存在NaN值，正在进行填充...")
        X_test = X_test.fillna(0)
    
    # 使用训练阶段的标准化器
    X_test_scaled = train_feature_info['scaler'].transform(X_test)
    
    # 预测（对数尺度）
    predictions_log = final_model.predict(X_test_scaled)
    
    # 转换回原始尺度
    predictions = np.expm1(predictions_log)
    
    # 确保预测值非负
    predictions = np.maximum(predictions, 0)
    
    # 5. 将预测结果与ID匹配
    prediction_df = pd.DataFrame({
        'ID': test_df_filtered['ID'].values,
        'price': predictions
    })
    
    # 6. 将预测结果匹配到模板中
    submission_filled = submission_df.merge(prediction_df, on='ID', how='left')
    
    # 检查是否有未匹配的ID
    unmatched_ids = submission_filled[submission_filled['price'].isna()]['ID']
    if len(unmatched_ids) > 0:
        print(f"警告: 有 {len(unmatched_ids)} 个ID未能匹配到预测结果")
        median_price = np.median(predictions)
        submission_filled['price'].fillna(median_price, inplace=True)
    
    # 7. 保存提交文件
    submission_filled[['ID', 'price']].to_csv(output_path, index=False)
    print(f"提交文件已保存: {output_path}")
    
    # 8. 显示预测统计信息
    print(f"\n预测结果统计:")
    print(f"  - 预测样本数: {len(predictions)}")
    print(f"  - 价格范围: {predictions.min():.2f} - {predictions.max():.2f}")
    print(f"  - 平均预测价格: {predictions.mean():.2f}")
    print(f"  - 负值数量: {np.sum(predictions < 0)}")
    
    return submission_filled

# 主执行代码
print("加载已经处理好的训练数据...")
train_df = pd.read_csv(r'C:\Users\lenovo\处理后的数据_综合特征.csv')
print(f"训练数据形状: {train_df.shape}")

# 检查目标变量的分布
print(f"\n目标变量(Price)统计:")
print(f"  最小值: {train_df['Price'].min():.2f}")
print(f"  最大值: {train_df['Price'].max():.2f}")
print(f"  平均值: {train_df['Price'].mean():.2f}")
print(f"  中位数: {train_df['Price'].median():.2f}")

# 检查是否有异常的价格值
price_q1 = train_df['Price'].quantile(0.01)
price_q99 = train_df['Price'].quantile(0.99)
print(f"  1%分位数: {price_q1:.2f}")
print(f"  99%分位数: {price_q99:.2f}")

print("\n准备训练特征和目标变量...")

# 准备训练数据
X, feature_names = prepare_features(train_df, is_training=True)
y = train_df['Price'].copy()

print(f"特征矩阵形状: {X.shape}")
print(f"目标变量形状: {y.shape}")

# 训练模型
results, final_feature_names = train_model(X, y)

# 打印性能表格
print_performance_table(results)

# 选择最佳模型
valid_results = {k: v for k, v in results.items() if v is not None and v['negative_count'] == 0}
if valid_results:
    best_model_name = min(valid_results.keys(), key=lambda x: valid_results[x]['test_mae'])
    best_result = valid_results[best_model_name]
    
    print(f"\n🎯 最佳模型: {best_model_name}")
    print(f"  样本内 MAE: {best_result['train_mae']:,.2f}")
    print(f"  样本外 MAE: {best_result['test_mae']:,.2f}")
    print(f"  交叉验证 MAE: {best_result['cv_mae']:,.2f}")
    print(f"  预测范围: {best_result['min_pred']:,.2f} - {best_result['max_pred']:,.2f}")
    
    final_model = best_result['model']
else:
    print("错误: 没有找到合适的模型")
    final_model = Ridge(alpha=10.0)

print("\n开始对外部测试集进行预测...")
test_data_path = r"C:\Users\lenovo\Desktop\期中数据\ruc_Class25Q2_test_price.csv"
template_path = r"C:\Users\lenovo\Desktop\期中数据\submission_template_Class25Q2.csv"
output_path = r"C:\Users\lenovo\Desktop\期中数据\submission_Class25Q2.csv"

# 进行预测
submission_result = predict_test_data(
    test_data_path, template_path, output_path, final_model, final_feature_names
)

print("\n" + "="*60)
print("外部测试集预测完成!")
print("="*60)

加载已经处理好的训练数据...
训练数据形状: (103871, 78)

目标变量(Price)统计:
  最小值: 74553.30
  最大值: 56226431.30
  平均值: 2262366.07
  中位数: 1479407.11
  1%分位数: 254478.05
  99%分位数: 12990538.99

准备训练特征和目标变量...
训练特征准备...
从原始数据创建特征...
  - 建筑面积处理完成，范围: 11.7 - 508.1
  - 套内面积处理完成，35984 个样本有套内面积
  - 绿化率处理完成，范围: 0.000 - 105.000
  - 容积率处理完成，范围: 0.000 - 30.000
  - 电梯信息处理完成，70830 个样本有电梯
  - 别墅信息处理完成，103871 个样本是别墅
  - 产权信息处理完成，103871 个样本是共有产权
  - 楼层信息处理完成，类型分布: {'中楼层': np.int64(36251), '高楼层': np.int64(32132), '低楼层': np.int64(30668), '顶层': np.int64(2264), '底层': np.int64(1854), '地下室': np.int64(702)}
  - 装修信息处理完成，类型分布: {'精装': np.int64(46585), '其他': np.int64(24400), '简装': np.int64(20522), '毛坯': np.int64(12364)}
成功加载地理聚类映射表，包含 876 个板块
  - 地理聚类应用完成，分布: {2.0: np.int64(65580), 1.0: np.int64(25535), 3.0: np.int64(8674), 0.0: np.int64(3210), 4.0: np.int64(658), -1.0: np.int64(214)}
训练特征矩阵形状: (103871, 27)
使用的特征数量: 27
特征矩阵形状: (103871, 27)
目标变量形状: (103871,)

训练模型...
  训练集样本数: 83096
  测试集样本数: 20775
  清理后训练样本数: 81434
  移除异常值: 1662 个

--- 训

In [12]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
import warnings
import re
warnings.filterwarnings('ignore')

# 全局变量
train_feature_info = {}

def create_rent_geoclustering(df, n_clusters_range=(3, 10), min_samples_per_plate=5):
    """
    基于租房数据板块价格特征的地理聚类
    返回聚类结果和详细的板块-聚类映射关系
    """
    print("="*60)
    print("开始基于租房数据板块价格的地理聚类分析")
    print("="*60)
    
    # 1. 计算每个板块的价格统计特征
    print("\n1. 计算板块租金统计特征...")
    plate_stats = df.groupby('板块').agg({
        'Price': ['mean', 'median', 'std', 'count', 'min', 'max']
    }).round(2)
    
    # 扁平化列名
    plate_stats.columns = ['价格均值', '价格中位数', '价格标准差', '样本数量', '最低价格', '最高价格']
    plate_stats = plate_stats.reset_index()
    
    # 2. 过滤样本量太少的板块
    print(f"\n2. 过滤样本量少于{min_samples_per_plate}的板块...")
    valid_plates = plate_stats[plate_stats['样本数量'] >= min_samples_per_plate].copy()
    invalid_plates = plate_stats[plate_stats['样本数量'] < min_samples_per_plate]['板块'].tolist()
    
    print(f"   - 总板块数: {len(plate_stats)}")
    print(f"   - 有效板块数: {len(valid_plates)}")
    print(f"   - 无效板块数(样本不足): {len(invalid_plates)}")
    
    if len(valid_plates) < 3:
        print("错误: 有效板块数量不足，无法进行聚类")
        return df, {}, {}
    
    # 3. 选择聚类特征并标准化
    print("\n3. 准备聚类特征...")
    clustering_features = ['价格均值', '价格标准差']  # 使用均值和标准差
    
    scaler = StandardScaler()
    X = scaler.fit_transform(valid_plates[clustering_features])
    
    # 4. 使用肘部法则确定最佳聚类数量
    print("\n4. 确定最佳聚类数量...")
    wcss = []
    k_range = range(n_clusters_range[0], min(n_clusters_range[1] + 1, len(valid_plates)))
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        wcss.append(kmeans.inertia_)
    
    # 自动选择拐点（肘部）
    optimal_k = find_elbow_point(wcss, k_range)
    print(f"   - 自动选择聚类数量: {optimal_k}")
    
    # 5. 执行K-means聚类
    print(f"\n5. 执行K-means聚类 (k={optimal_k})...")
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    
    # 6. 创建板块到聚类的映射
    plate_to_cluster = dict(zip(valid_plates['板块'], cluster_labels))
    
    # 7. 应用到原始数据
    print("\n6. 将聚类结果应用到原始数据...")
    df_result = df.copy()
    df_result['板块价格聚类'] = df_result['板块'].map(plate_to_cluster)
    df_result['板块价格聚类'] = df_result['板块价格聚类'].fillna(-1)  # 无效板块标记为-1
    
    # 8. 创建详细的聚类映射记录
    print("\n7. 创建详细的聚类映射记录...")
    cluster_mapping_details = {}
    cluster_plate_lists = {}
    
    # 为每个聚类创建板块列表
    for cluster_id in range(optimal_k):
        plates_in_cluster = [plate for plate, cluster in plate_to_cluster.items() if cluster == cluster_id]
        cluster_plate_lists[f'聚类_{cluster_id}'] = plates_in_cluster
        cluster_mapping_details[f'聚类_{cluster_id}'] = {
            '板块数量': len(plates_in_cluster),
            '板块列表': plates_in_cluster,
            '平均价格': valid_plates[valid_plates['板块'].isin(plates_in_cluster)]['价格均值'].mean()
        }
    
    # 添加无效板块记录
    cluster_plate_lists['无效板块(样本不足)'] = invalid_plates
    cluster_mapping_details['无效板块'] = {
        '板块数量': len(invalid_plates),
        '板块列表': invalid_plates,
        '平均价格': 'N/A'
    }
    
    return df_result, cluster_mapping_details, cluster_plate_lists

def find_elbow_point(wcss, k_range):
    """自动找到肘部拐点"""
    # 计算二阶差分找到最大曲率点
    differences = []
    for i in range(1, len(wcss)-1):
        diff = (wcss[i-1] - wcss[i]) - (wcss[i] - wcss[i+1])
        differences.append(diff)
    
    if differences:
        elbow_index = differences.index(max(differences)) + 1
        return k_range[elbow_index]
    else:
        # 如果没有找到明显拐点，选择中间值
        return k_range[len(k_range)//2]

def load_rent_geocluster_mapping():
    """加载租房地理聚类映射表"""
    try:
        mapping_df = pd.read_csv('租房板块聚类简化映射表.csv')
        print(f"成功加载租房地理聚类映射表，包含 {len(mapping_df)} 个板块")
        return mapping_df
    except FileNotFoundError:
        print("警告: 未找到租房地理聚类映射表，将使用默认聚类")
        return None

def extract_numeric_from_string(s):
    """从字符串中提取数值"""
    if pd.isna(s):
        return 0
    # 匹配数字，包括小数
    numbers = re.findall(r'\d+\.?\d*', str(s))
    if numbers:
        return float(numbers[0])
    return 0

def process_building_area(area_str):
    """处理面积，去掉单位并转换为数值"""
    if pd.isna(area_str):
        return 0
    # 去掉"㎡"单位并提取数值
    area_str = str(area_str).replace('㎡', '').strip()
    return extract_numeric_from_string(area_str)

def process_percentage(percent_str):
    """处理百分比数值"""
    if pd.isna(percent_str):
        return 0
    percent_str = str(percent_str).replace('%', '').strip()
    return extract_numeric_from_string(percent_str)

def process_floor(floor_str):
    """处理楼层信息，去掉括号说明并分类"""
    if pd.isna(floor_str):
        return '其他'
    
    # 去掉括号及后面的内容
    floor_str = str(floor_str).split('/')[0].strip().lower()  # 租房数据格式不同
    
    if '地下室' in floor_str or '地下' in floor_str:
        return '地下室'
    elif '底层' in floor_str or '底楼' in floor_str or '1/' in floor_str:
        return '底层'
    elif '低楼层' in floor_str or '低层' in floor_str:
        return '低楼层'
    elif '中楼层' in floor_str or '中层' in floor_str:
        return '中楼层'
    elif '高楼层' in floor_str or '高层' in floor_str:
        return '高楼层'
    elif '顶层' in floor_str or '顶楼' in floor_str:
        return '顶层'
    else:
        return '其他'

def process_renovation(reno_str):
    """处理装修情况"""
    if pd.isna(reno_str):
        return '其他'
    
    reno_str = str(reno_str).lower()
    if '精装' in reno_str:
        return '精装'
    elif '简装' in reno_str:
        return '简装'
    elif '毛坯' in reno_str:
        return '毛坯'
    elif '豪华' in reno_str:
        return '豪华装修'
    elif '中等' in reno_str:
        return '中等装修'
    else:
        return '其他'

def create_rent_features_from_raw_data(df, is_training=True):
    """从原始租房数据创建特征"""
    processed_df = df.copy()
    
    print("从原始租房数据创建特征...")
    
    # 1. 处理面积（关键步骤！）
    if '面积' in processed_df.columns:
        processed_df['面积_数值'] = processed_df['面积'].apply(process_building_area)
        print(f"  - 面积处理完成，范围: {processed_df['面积_数值'].min():.1f} - {processed_df['面积_数值'].max():.1f}")
    else:
        processed_df['面积_数值'] = 0
        print("  - 警告: 未找到面积列")
    
    # 2. 处理绿化率
    if '绿化率' in processed_df.columns:
        processed_df['绿化率_数值'] = processed_df['绿化率'].apply(process_percentage) / 100
        print(f"  - 绿化率处理完成，范围: {processed_df['绿化率_数值'].min():.3f} - {processed_df['绿化率_数值'].max():.3f}")
    elif '绿 化 率' in processed_df.columns:
        processed_df['绿化率_数值'] = processed_df['绿 化 率'].apply(process_percentage) / 100
        print(f"  - 绿化率处理完成，范围: {processed_df['绿化率_数值'].min():.3f} - {processed_df['绿化率_数值'].max():.3f}")
    else:
        processed_df['绿化率_数值'] = 0
        print("  - 警告: 未找到绿化率列")
    
    # 3. 处理容积率
    if '容积率' in processed_df.columns:
        processed_df['容积率_数值'] = processed_df['容积率'].apply(extract_numeric_from_string)
        print(f"  - 容积率处理完成，范围: {processed_df['容积率_数值'].min():.3f} - {processed_df['容积率_数值'].max():.3f}")
    elif '容 积 率' in processed_df.columns:
        processed_df['容积率_数值'] = processed_df['容 积 率'].apply(extract_numeric_from_string)
        print(f"  - 容积率处理完成，范围: {processed_df['容积率_数值'].min():.3f} - {processed_df['容积率_数值'].max():.3f}")
    else:
        processed_df['容积率_数值'] = 0
        print("  - 警告: 未找到容积率列")
    
    # 4. 处理是否有电梯
    if '电梯' in processed_df.columns:
        processed_df['有电梯'] = processed_df['电梯'].astype(str).str.contains('有|是|True|true|1', na=False).astype(int)
        elevator_count = processed_df['有电梯'].sum()
        print(f"  - 电梯信息处理完成，{elevator_count} 个样本有电梯")
    else:
        processed_df['有电梯'] = 0
        print("  - 警告: 未找到电梯信息列")
    
    # 5. 处理是否有车位
    if '车位' in processed_df.columns:
        processed_df['有车位'] = (~processed_df['车位'].isna() & (processed_df['车位'] != '')).astype(int)
        parking_count = processed_df['有车位'].sum()
        print(f"  - 车位信息处理完成，{parking_count} 个样本有车位")
    else:
        processed_df['有车位'] = 0
        print("  - 警告: 未找到车位信息列")
    
    # 6. 处理是否有燃气
    if '燃气' in processed_df.columns:
        processed_df['有燃气'] = processed_df['燃气'].astype(str).str.contains('有|是|True|true|1', na=False).astype(int)
        gas_count = processed_df['有燃气'].sum()
        print(f"  - 燃气信息处理完成，{gas_count} 个样本有燃气")
    else:
        processed_df['有燃气'] = 0
        print("  - 警告: 未找到燃气信息列")
    
    # 7. 处理租赁方式
    if '租赁方式' in processed_df.columns:
        # 创建租赁方式哑变量
        rent_type_dummies = pd.get_dummies(processed_df['租赁方式'].astype(str), prefix='租赁方式')
        for col in rent_type_dummies.columns:
            processed_df[col] = rent_type_dummies[col]
        print(f"  - 租赁方式处理完成，类型分布: {dict(processed_df['租赁方式'].value_counts())}")
    else:
        processed_df['租赁方式_整租'] = 1  # 默认整租
        print("  - 警告: 未找到租赁方式列，默认整租")
    
    # 8. 处理付款方式
    if '付款方式' in processed_df.columns:
        # 创建付款方式哑变量
        payment_dummies = pd.get_dummies(processed_df['付款方式'].astype(str), prefix='付款方式')
        for col in payment_dummies.columns:
            processed_df[col] = payment_dummies[col]
        print(f"  - 付款方式处理完成，类型分布: {dict(processed_df['付款方式'].value_counts())}")
    else:
        processed_df['付款方式_季付价'] = 1  # 默认季付
        print("  - 警告: 未找到付款方式列，默认季付")
    
    # 9. 处理楼层信息
    if '楼层' in processed_df.columns:
        processed_df['楼层类型'] = processed_df['楼层'].apply(process_floor)
        # 创建楼层哑变量
        all_floor_types = ['地下室', '底层', '低楼层', '中楼层', '高楼层', '顶层', '其他']
        floor_dummies = pd.get_dummies(processed_df['楼层类型'], prefix='楼层')
        
        # 确保所有楼层类型都有对应的列
        for floor_type in all_floor_types:
            col_name = f'楼层_{floor_type}'
            if col_name in floor_dummies.columns:
                processed_df[col_name] = floor_dummies[col_name]
            else:
                processed_df[col_name] = 0
        
        print(f"  - 楼层信息处理完成，类型分布: {dict(processed_df['楼层类型'].value_counts())}")
    else:
        processed_df['楼层类型'] = '其他'
        all_floor_types = ['地下室', '底层', '低楼层', '中楼层', '高楼层', '顶层', '其他']
        for floor_type in all_floor_types:
            processed_df[f'楼层_{floor_type}'] = 0
        print("  - 警告: 未找到楼层信息列")
    
    # 10. 处理装修情况
    if '装修' in processed_df.columns:
        processed_df['装修类型'] = processed_df['装修'].apply(process_renovation)
        # 创建装修哑变量
        all_reno_types = ['精装', '简装', '毛坯', '豪华装修', '中等装修', '其他']
        reno_dummies = pd.get_dummies(processed_df['装修类型'], prefix='装修')
        
        # 确保所有装修类型都有对应的列
        for reno_type in all_reno_types:
            col_name = f'装修_{reno_type}'
            if col_name in reno_dummies.columns:
                processed_df[col_name] = reno_dummies[col_name]
            else:
                processed_df[col_name] = 0
        
        print(f"  - 装修信息处理完成，类型分布: {dict(processed_df['装修类型'].value_counts())}")
    else:
        processed_df['装修类型'] = '其他'
        all_reno_types = ['精装', '简装', '毛坯', '豪华装修', '中等装修', '其他']
        for reno_type in all_reno_types:
            processed_df[f'装修_{reno_type}'] = 0
        print("  - 警告: 未找到装修信息列")
    
    # 11. 处理户型
    if '户型' in processed_df.columns:
        # 提取卧室数量
        processed_df['卧室数量'] = processed_df['户型'].astype(str).str.extract(r'(\d+)室')[0].fillna(1).astype(int)
        print(f"  - 户型处理完成，卧室数量范围: {processed_df['卧室数量'].min()} - {processed_df['卧室数量'].max()}")
    else:
        processed_df['卧室数量'] = 1
        print("  - 警告: 未找到户型列，默认1室")
    
    # 12. 处理朝向
    if '朝向' in processed_df.columns:
        # 创建朝向哑变量
        direction_dummies = pd.get_dummies(processed_df['朝向'].astype(str), prefix='朝向')
        for col in direction_dummies.columns:
            processed_df[col] = direction_dummies[col]
        print(f"  - 朝向处理完成，主要朝向: {processed_df['朝向'].value_counts().head(3).to_dict()}")
    else:
        processed_df['朝向_南'] = 1  # 默认南向
        print("  - 警告: 未找到朝向列，默认南向")
    
    # 13. 应用租房地理聚类
    mapping_df = load_rent_geocluster_mapping()
    if mapping_df is not None and '板块' in processed_df.columns:
        plate_to_cluster = dict(zip(mapping_df['板块'], mapping_df['聚类ID']))
        processed_df['地理聚类'] = processed_df['板块'].map(plate_to_cluster)
        processed_df['地理聚类'] = processed_df['地理聚类'].fillna(-1)
        
        # 创建聚类哑变量
        cluster_ids = mapping_df['聚类ID'].unique()
        for cluster_id in cluster_ids:
            processed_df[f'地理聚类_{cluster_id}'] = (processed_df['地理聚类'] == cluster_id).astype(int)
        
        cluster_counts = processed_df['地理聚类'].value_counts()
        print(f"  - 地理聚类应用完成，分布: {dict(cluster_counts)}")
    else:
        print("  - 警告: 无法应用地理聚类")
        # 如果没有聚类映射，创建空的聚类哑变量
        for cluster_id in range(5):
            processed_df[f'地理聚类_{cluster_id}'] = 0
    
    return processed_df

def prepare_rent_features(df, is_training=True):
    """准备租房特征矩阵"""
    print(f"{'训练' if is_training else '测试'}特征准备...")
    
    # 从原始数据创建特征
    df_processed = create_rent_features_from_raw_data(df, is_training)
    
    # 定义特征列
    feature_columns = []
    
    # 1. 地理聚类特征
    geo_cluster_features = [f for f in df_processed.columns if f.startswith('地理聚类_')]
    feature_columns.extend(geo_cluster_features)
    
    # 2. 面积特征
    area_features = ['面积_数值', '卧室数量']
    feature_columns.extend([f for f in area_features if f in df_processed.columns])
    
    # 3. 数值特征
    numeric_features = ['绿化率_数值', '容积率_数值']
    feature_columns.extend([f for f in numeric_features if f in df_processed.columns])
    
    # 4. 布尔特征
    bool_features = ['有电梯', '有车位', '有燃气']
    feature_columns.extend([f for f in bool_features if f in df_processed.columns])
    
    # 5. 楼层哑变量
    floor_features = [f for f in df_processed.columns if f.startswith('楼层_')]
    feature_columns.extend(floor_features)
    
    # 6. 装修哑变量
    reno_features = [f for f in df_processed.columns if f.startswith('装修_')]
    feature_columns.extend(reno_features)
    
    # 7. 租赁方式哑变量
    rent_type_features = [f for f in df_processed.columns if f.startswith('租赁方式_')]
    feature_columns.extend(rent_type_features)
    
    # 8. 付款方式哑变量
    payment_features = [f for f in df_processed.columns if f.startswith('付款方式_')]
    feature_columns.extend(payment_features)
    
    # 9. 朝向哑变量
    direction_features = [f for f in df_processed.columns if f.startswith('朝向_')]
    feature_columns.extend(direction_features)
    
    # 创建特征矩阵
    X = pd.DataFrame(index=df_processed.index)
    
    # 确保所有特征都存在
    for feature in feature_columns:
        if feature in df_processed.columns:
            X[feature] = df_processed[feature]
        else:
            print(f"警告: 特征 '{feature}' 不存在，将用0填充")
            X[feature] = 0
    
    # 确保所有特征都是数值类型
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        if X[col].isna().any():
            X[col].fillna(0, inplace=True)
    
    print(f"{'训练' if is_training else '测试'}特征矩阵形状: {X.shape}")
    print(f"使用的特征数量: {len(feature_columns)}")
    
    return X, feature_columns

def train_rent_model(X, y):
    """训练租房模型"""
    print("\n训练租房模型...")
    
    # 1. 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=111
    )
    
    print(f"  训练集样本数: {len(X_train)}")
    print(f"  测试集样本数: {len(X_test)}")
    
    # 2. 处理异常值
    price_lower = y_train.quantile(0.01)
    price_upper = y_train.quantile(0.99)
    
    valid_indices = (y_train >= price_lower) & (y_train <= price_upper)
    X_train_clean = X_train[valid_indices].copy()
    y_train_clean = y_train[valid_indices].copy()
    
    print(f"  清理后训练样本数: {len(y_train_clean)}")
    print(f"  移除异常值: {len(y_train) - len(y_train_clean)} 个")
    
    # 3. 对目标变量进行对数变换
    y_train_log = np.log1p(y_train_clean)
    y_test_log = np.log1p(y_test)
    
    # 4. 标准化特征
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled = scaler.transform(X_test)
    
    # 保存标准化器
    train_feature_info['scaler'] = scaler
    
    # 5. 定义模型
    models = {
        'OLS': LinearRegression(),
        'Ridge_0.1': Ridge(alpha=0.1),
        'Ridge_1.0': Ridge(alpha=1.0),
        'Ridge_10.0': Ridge(alpha=10.0),
        'LASSO_0.1': Lasso(alpha=0.1, max_iter=10000),
        'LASSO_1.0': Lasso(alpha=1.0, max_iter=10000),
        'ElasticNet_0.1': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
        'ElasticNet_1.0': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
    }
    
    results = {}
    
    # 6. 训练和评估模型
    for name, model in models.items():
        print(f"\n--- 训练 {name} 模型 ---")
        
        try:
            # 训练模型
            model.fit(X_train_scaled, y_train_log)
            
            # 样本内预测
            y_train_pred_log = model.predict(X_train_scaled)
            y_train_pred = np.expm1(y_train_pred_log)
            
            # 样本外预测
            y_test_pred_log = model.predict(X_test_scaled)
            y_test_pred = np.expm1(y_test_pred_log)
            
            # 计算指标
            train_mae = mean_absolute_error(y_train_clean, y_train_pred)
            train_r2 = r2_score(y_train_clean, y_train_pred)
            
            test_mae = mean_absolute_error(y_test, y_test_pred)
            test_r2 = r2_score(y_test, y_test_pred)
            
            # 6折交叉验证
            cv_scores = cross_val_score(
                model, X_train_scaled, y_train_log, 
                cv=6, scoring='neg_mean_absolute_error'
            )
            cv_mae = -cv_scores.mean()
            cv_std = cv_scores.std()
            
            # 检查预测值
            min_pred = min(y_train_pred.min(), y_test_pred.min())
            max_pred = max(y_train_pred.max(), y_test_pred.max())
            negative_count = np.sum(y_test_pred < 0)
            
            results[name] = {
                'model': model,
                'train_mae': train_mae,
                'train_r2': train_r2,
                'test_mae': test_mae,
                'test_r2': test_r2,
                'cv_mae': cv_mae,
                'cv_std': cv_std,
                'min_pred': min_pred,
                'max_pred': max_pred,
                'negative_count': negative_count
            }
            
            print(f"  样本内 MAE: {train_mae:,.2f}")
            print(f"  样本外 MAE: {test_mae:,.2f}")
            print(f"  交叉验证 MAE: {cv_mae:,.2f} ± {cv_std:,.2f}")
            print(f"  预测范围: {min_pred:,.2f} - {max_pred:,.2f}")
            print(f"  负值数量: {negative_count}")
            
        except Exception as e:
            print(f"  {name} 模型训练失败: {e}")
            results[name] = None
    
    return results, X_train_clean.columns.tolist()

def print_rent_performance_table(results):
    """打印租房性能表格"""
    print("\n" + "="*80)
    print("租房性能汇总表格 (MAE指标)")
    print("="*80)
    print("| Metrics           | In sample     | out of sample | Cross-validation | Kaggle Score |")
    print("|-------------------|---------------|---------------|------------------|--------------|")
    
    # 确定最佳模型
    valid_results = {k: v for k, v in results.items() if v is not None and v['negative_count'] == 0}
    best_model_name = min(valid_results.keys(), key=lambda x: valid_results[x]['test_mae']) if valid_results else None
    
    # 主要模型显示
    main_models = {
        'OLS': 'OLS',
        'LASSO': 'LASSO_0.1',
        'Best Linear Model': best_model_name
    }
    
    for display_name, model_key in main_models.items():
        if model_key and model_key in results and results[model_key] is not None:
            result = results[model_key]
            print(f"| {display_name:17} | {result['train_mae']:12,.2f} | {result['test_mae']:12,.2f} | {result['cv_mae']:15,.2f} | {'-':12} |")
        else:
            print(f"| {display_name:17} | {'N/A':12} | {'N/A':12} | {'N/A':15} | {'-':12} |")
    
    print("="*80)

def save_rent_clustering_results(df_with_clusters, cluster_mapping_details, cluster_plate_lists):
    """保存租房聚类结果和映射关系"""
    print("\n" + "="*60)
    print("保存租房聚类结果")
    print("="*60)
    
    # 1. 保存带聚类标签的完整数据
    df_with_clusters.to_csv('带租房板块聚类标签的完整数据.csv', index=False, encoding='utf-8-sig')
    print("✓ 保存带租房聚类标签的完整数据: '带租房板块聚类标签的完整数据.csv'")
    
    # 2. 保存详细的聚类映射关系
    mapping_df = pd.DataFrame.from_dict(cluster_mapping_details, orient='index')
    mapping_df.to_csv('租房板块聚类映射详情.csv', encoding='utf-8-sig')
    print("✓ 保存租房聚类映射详情: '租房板块聚类映射详情.csv'")
    
    # 3. 保存每个聚类的板块列表（便于后续调用）
    plate_lists_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in cluster_plate_lists.items()]))
    plate_lists_df.to_csv('各租房聚类板块列表.csv', index=False, encoding='utf-8-sig')
    print("✓ 保存各租房聚类板块列表: '各租房聚类板块列表.csv'")
    
    # 4. 创建简化的映射表（便于预测时使用）
    simple_mapping = []
    for cluster_name, details in cluster_mapping_details.items():
        if cluster_name != '无效板块':
            for plate in details['板块列表']:
                simple_mapping.append({
                    '板块': plate,
                    '聚类标签': cluster_name,
                    '聚类ID': int(cluster_name.split('_')[1])
                })
    
    simple_mapping_df = pd.DataFrame(simple_mapping)
    simple_mapping_df.to_csv('租房板块聚类简化映射表.csv', index=False, encoding='utf-8-sig')
    print("✓ 保存租房简化映射表: '租房板块聚类简化映射表.csv'")
    
    return simple_mapping_df

def predict_rent_test_data(test_path, template_path, output_path, final_model, feature_names):
    """预测租房测试集数据"""
    print("\n处理租房外部测试集...")
    
    # 1. 读取测试数据和提交模板
    test_df = pd.read_csv(test_path)
    submission_df = pd.read_csv(template_path)
    
    print(f"测试数据形状: {test_df.shape}")
    print(f"提交模板形状: {submission_df.shape}")
    
    # 2. 根据提交模板中的ID筛选需要预测的测试数据
    template_ids = set(submission_df['ID'])
    test_df_filtered = test_df[test_df['ID'].isin(template_ids)].copy()
    
    print(f"筛选后需要预测的测试数据形状: {test_df_filtered.shape}")
    
    # 3. 准备测试特征
    X_test, _ = prepare_rent_features(test_df_filtered, is_training=False)
    
    # 确保特征一致
    for feature in feature_names:
        if feature not in X_test.columns:
            print(f"警告: 特征 '{feature}' 在测试集中不存在，将用0填充")
            X_test[feature] = 0
    
    X_test = X_test[feature_names]
    
    # 4. 使用训练好的模型进行预测
    print("使用训练好的模型进行预测...")
    
    # 确保测试数据没有NaN值
    if X_test.isna().any().any():
        print("警告: 测试数据中存在NaN值，正在进行填充...")
        X_test = X_test.fillna(0)
    
    # 使用训练阶段的标准化器
    X_test_scaled = train_feature_info['scaler'].transform(X_test)
    
    # 预测（对数尺度）
    predictions_log = final_model.predict(X_test_scaled)
    
    # 转换回原始尺度
    predictions = np.expm1(predictions_log)
    
    # 确保预测值非负
    predictions = np.maximum(predictions, 0)
    
    # 5. 将预测结果与ID匹配
    prediction_df = pd.DataFrame({
        'ID': test_df_filtered['ID'].values,
        'price': predictions
    })
    
    # 6. 将预测结果匹配到模板中
    submission_filled = submission_df.merge(prediction_df, on='ID', how='left')
    
    # 检查是否有未匹配的ID
    unmatched_ids = submission_filled[submission_filled['price'].isna()]['ID']
    if len(unmatched_ids) > 0:
        print(f"警告: 有 {len(unmatched_ids)} 个ID未能匹配到预测结果")
        median_price = np.median(predictions)
        submission_filled['price'].fillna(median_price, inplace=True)
    
    # 7. 保存提交文件
    submission_filled[['ID', 'price']].to_csv(output_path, index=False)
    print(f"提交文件已保存: {output_path}")
    
    # 8. 显示预测统计信息
    print(f"\n预测结果统计:")
    print(f"  - 预测样本数: {len(predictions)}")
    print(f"  - 价格范围: {predictions.min():.2f} - {predictions.max():.2f}")
    print(f"  - 平均预测价格: {predictions.mean():.2f}")
    print(f"  - 负值数量: {np.sum(predictions < 0)}")
    
    return submission_filled

# 主执行代码
print("加载租房训练数据并进行地理聚类...")
rent_train_df = pd.read_csv(r"C:\Users\lenovo\Desktop\期中数据\ruc_Class25Q2_train_rent.csv")
print(f"租房训练数据形状: {rent_train_df.shape}")

# 对租房数据进行地理聚类
rent_df_with_clusters, rent_cluster_mapping_details, rent_cluster_plate_lists = create_rent_geoclustering(
    rent_train_df, n_clusters_range=(4, 8), min_samples_per_plate=5
)

# 保存租房聚类结果
rent_simple_mapping_df = save_rent_clustering_results(
    rent_df_with_clusters, rent_cluster_mapping_details, rent_cluster_plate_lists
)

# 检查目标变量的分布
print(f"\n目标变量(Price)统计:")
print(f"  最小值: {rent_train_df['Price'].min():.2f}")
print(f"  最大值: {rent_train_df['Price'].max():.2f}")
print(f"  平均值: {rent_train_df['Price'].mean():.2f}")
print(f"  中位数: {rent_train_df['Price'].median():.2f}")

# 检查是否有异常的价格值
price_q1 = rent_train_df['Price'].quantile(0.01)
price_q99 = rent_train_df['Price'].quantile(0.99)
print(f"  1%分位数: {price_q1:.2f}")
print(f"  99%分位数: {price_q99:.2f}")

print("\n准备租房训练特征和目标变量...")

# 准备训练数据
X, feature_names = prepare_rent_features(rent_df_with_clusters, is_training=True)
y = rent_df_with_clusters['Price'].copy()

print(f"特征矩阵形状: {X.shape}")
print(f"目标变量形状: {y.shape}")

# 训练模型
results, final_feature_names = train_rent_model(X, y)

# 打印性能表格
print_rent_performance_table(results)

# 选择最佳模型
valid_results = {k: v for k, v in results.items() if v is not None and v['negative_count'] == 0}
if valid_results:
    best_model_name = min(valid_results.keys(), key=lambda x: valid_results[x]['test_mae'])
    best_result = valid_results[best_model_name]
    
    print(f"\n🎯 最佳模型: {best_model_name}")
    print(f"  样本内 MAE: {best_result['train_mae']:,.2f}")
    print(f"  样本外 MAE: {best_result['test_mae']:,.2f}")
    print(f"  交叉验证 MAE: {best_result['cv_mae']:,.2f}")
    print(f"  预测范围: {best_result['min_pred']:,.2f} - {best_result['max_pred']:,.2f}")
    
    final_model = best_result['model']
else:
    print("错误: 没有找到合适的模型")
    final_model = Ridge(alpha=10.0)

print("\n开始对租房外部测试集进行预测...")
test_data_path = r"C:\Users\lenovo\Desktop\期中数据\ruc_Class25Q2_test_rent.csv"
template_path = r"C:\Users\lenovo\Desktop\期中数据\submission_template_Class25Q2.csv"
output_path = r"C:\Users\lenovo\Desktop\期中数据\submission_Class25Q2_rent.csv"

# 进行预测
submission_result = predict_rent_test_data(
    test_data_path, template_path, output_path, final_model, final_feature_names
)

print("\n" + "="*60)
print("租房外部测试集预测完成!")
print("="*60)

加载租房训练数据并进行地理聚类...
租房训练数据形状: (98899, 46)
开始基于租房数据板块价格的地理聚类分析

1. 计算板块租金统计特征...

2. 过滤样本量少于5的板块...
   - 总板块数: 916
   - 有效板块数: 835
   - 无效板块数(样本不足): 81

3. 准备聚类特征...

4. 确定最佳聚类数量...
   - 自动选择聚类数量: 5

5. 执行K-means聚类 (k=5)...

6. 将聚类结果应用到原始数据...

7. 创建详细的聚类映射记录...

保存租房聚类结果
✓ 保存带租房聚类标签的完整数据: '带租房板块聚类标签的完整数据.csv'
✓ 保存租房聚类映射详情: '租房板块聚类映射详情.csv'
✓ 保存各租房聚类板块列表: '各租房聚类板块列表.csv'
✓ 保存租房简化映射表: '租房板块聚类简化映射表.csv'

目标变量(Price)统计:
  最小值: 17938.07
  最大值: 15404193.15
  平均值: 582908.98
  中位数: 394936.89
  1%分位数: 92972.67
  99%分位数: 3227491.22

准备租房训练特征和目标变量...
训练特征准备...
从原始租房数据创建特征...
  - 面积处理完成，范围: 6.0 - 440.0
  - 绿化率处理完成，范围: 0.000 - 105.000
  - 容积率处理完成，范围: 0.000 - 30.000
  - 电梯信息处理完成，69234 个样本有电梯
  - 车位信息处理完成，24764 个样本有车位
  - 燃气信息处理完成，84996 个样本有燃气
  - 租赁方式处理完成，类型分布: {'整租': np.int64(93009), '合租': np.int64(5890)}
  - 付款方式处理完成，类型分布: {'季付价': np.int64(54530), '月付价': np.int64(20955), '半年付价': np.int64(4194), '年付价': np.int64(748), '双月付价': np.int64(44), 'https://img.ljcdn.com/usercent': np.int64(3), 'https://image